# フレーズ分類ラベルアノテーションと作業者間一致率

書籍 第5章「言語資源のつくり方」（5-2 仕様・品質管理・アノテーションの外注、5-6 演習：フレーズ分類ラベルアノテーション）に対応する演習ノートブックです。
宿泊レビューから抽出したフレーズに対する評価極性のアノテーションを題材に、以下を実際に手を動かして確認します。

1. アノテーションシートの設計（図 5.4 準拠）と 5値／3値のラベル粒度（5-6-1, 5-6-2）
2. 作業者間一致率（Cohen の $\kappa$）の計算と不一致事例の分析（5-6「品質管理」）
3. 仕様書（ガイドライン）の改善と未知データ（第2ラウンド）での再測定（5-2「仕様」、5-6「品質管理」）
4. 3人以上のアノテーション（Fleiss の $\kappa$）とクラウドソーシングの品質管理（5-2「品質管理」「アノテーションの外注」）
5. 欠損値や順序尺度を扱える一致率指標 **Krippendorff の $\alpha$**（名義尺度 vs 順序尺度・欠損あり行列）（5-2 補足）

## 1. 準備とデータの確認

第5章用に作成したサンプルデータ `phrase-annotation.csv`（100件の宿泊レビューフレーズ、第1ラウンド 50件 + 第2ラウンド 50件）をダウンロードして読み込みます。

> **補足**: Excel や Google スプレッドシート上でのタグ付け作業をそのまま試したい場合は、同じ 100 件を図 5.4 と同じ 4 列（`フレーズID` / `カテゴリ` / `極性` / `フレーズ`、列固定・5段階矢印アイコンの条件付き書式付き）で収録した [`misc/phrase-annotation.xlsx`](https://github.com/mamorlis/nlpbook/raw/main/misc/phrase-annotation.xlsx) もリポジトリに用意しています。

In [1]:
!curl -fsSLO https://raw.githubusercontent.com/mamorlis/nlpbook/refs/heads/main/misc/phrase-annotation.csv

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

df = pd.read_csv("phrase-annotation.csv").fillna({"コメント": ""})
print(f"総件数: {len(df)} 件")
print("ラウンド別件数:", df["ラウンド"].value_counts().sort_index().to_dict())
print("5段階評価『極性』(1〜5) の分布:", df["極性"].value_counts().sort_index().to_dict())
print("3値評価『極性ラベル』の分布:", df["極性ラベル"].value_counts().to_dict())
df[["フレーズID", "ラウンド", "カテゴリ", "極性", "作業者A", "作業者B", "極性ラベル", "フレーズ", "コメント"]].head(10)

総件数: 100 件
ラウンド別件数: {1: 50, 2: 50}
5段階評価『極性』(1〜5) の分布: {1: 18, 2: 12, 3: 32, 4: 10, 5: 28}
3値評価『極性ラベル』の分布: {'pos': 38, 'neu': 32, 'neg': 30}


,フレーズID,ラウンド,カテゴリ,極性,作業者A,作業者B,極性ラベル,フレーズ,コメント
0,1,1,[動詞P],5,pos,neg,pos,大通り 沿い で 車 の 音 は 聞こえ ま す が 二重 窓 の おかげ で 気 に なら...,逆接表現：騒音への言及をBがneg、解消された点をAがposと判定
1,2,1,[助動詞でP],5,pos,neg,pos,外観 は 地味 な 印象 で し た が 内装 は 綺麗 に リニューアル 済み で す,逆接表現：前件のマイナスと後件のプラスで極性が反転
2,3,1,[助詞がP],3,pos,neu,neu,駅 の 近く に 飲食店 が 多 く あ る の が,A:立地の良さでpos / B:周辺状況の記述のためneu
3,4,1,[助詞もP],3,pos,neu,neu,天然 温泉 の 大 浴場 も,A:魅力的な設備でpos / B:名詞句のみで判断保留しneu
4,5,1,[名詞P],5,pos,pos,pos,価格 以上 の 大 満足,
5,6,1,[助動詞でP],1,neg,neg,neg,隣室 の ドア の 開閉 音 が うるさ く て ストレス で し た,
6,7,1,[動詞P],3,neu,pos,neu,値段 相応 の 設備 で 特に 過不足 は あり ま せ ん,A:どちらでもない評価でneu / B:不満なしを肯定とみてpos
7,8,1,[動詞P],3,pos,neu,neu,ホテル の すぐ 隣 に コンビニ も あ り 、,A:利便性を評価しpos / B:設備有無の記述のためneu
8,9,1,[助詞をP],3,neu,neu,neu,チェック イン 手続き を,
9,10,1,[助詞でP],3,pos,neu,neu,２４ 時間 営業 の フロント 対応 で,A:安心感がありpos / B:営業形態の記述のためneu


### アノテーション粒度とシート設計のポイント（5-6-1, 5-6-2）

- **ラベルの粒度**: 5段階（1〜5）の分布を見ると、中間の **2** や **4** に比べて **1・3・5** の飛び飛びの値に集中しやすいことがわかります（5-6-1）。このように実質的な判断が 3 値に近い場合は、3値のマルチクラス（`pos` / `neu` / `neg`）に整理する方が作業者間のばらつきを抑えやすくなります。
- **ユニークな `フレーズID` を付与する**: 不一致事例を議論したり元のレビュー文脈へ戻したりするときに必須です。
- **補助情報（句構造解析の `カテゴリ` など）を付ける**: 直接の付与対象でなくても人手判断の手がかりになります。
- **ラベル列を先頭付近に固定表示し、入力規則（プルダウン）や条件付き書式を設定する**: スクロールしても見やすく、不正な値の混入を防げます。
- **`コメント` 欄を用意する**: 迷った事例や判断理由をメモしておくと、後からの仕様見直しや事例紹介に役立ちます。

---

## 2. 作業者間一致率（Cohen の $\kappa$）の計算

### 2-1. 2値分類の練習問題（5-6「品質管理」）

まず、書籍 5-6「品質管理」の最初の練習問題（50事例、`pos` / `neg` の2値分類）について、定義式どおりに単純一致率 $p_o$、偶然の一致率 $p_e$、および Cohen の $\kappa$ 係数

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

を計算し、`scikit-learn` の `cohen_kappa_score` と一致することを確認します。

In [3]:
def compute_cohen_kappa(y_a, y_b, labels):
    """混同行列から単純一致率 p_o、偶然一致率 p_e、Cohen の κ を計算する"""
    cm = pd.crosstab(pd.Series(y_a, name="作業者A"), pd.Series(y_b, name="作業者B"))
    cm = cm.reindex(index=labels, columns=labels, fill_value=0)
    n = cm.to_numpy().sum()
    p_o = np.trace(cm.to_numpy()) / n
    p_a = cm.sum(axis=1).to_numpy() / n
    p_b = cm.sum(axis=0).to_numpy() / n
    p_e = float(np.sum(p_a * p_b))
    kappa = float((p_o - p_e) / (1.0 - p_e))
    return cm, float(p_o), p_e, kappa

# 書籍 5-6「品質管理」の2値分類の混同行列を再現（pos/pos=20, pos/neg=5, neg/pos=10, neg/neg=15）
a_bin = ["pos"] * 20 + ["pos"] * 5 + ["neg"] * 10 + ["neg"] * 15
b_bin = ["pos"] * 20 + ["neg"] * 5 + ["pos"] * 10 + ["neg"] * 15

cm_bin, po_bin, pe_bin, k_bin = compute_cohen_kappa(a_bin, b_bin, ["pos", "neg"])
display(cm_bin)
print(f"p_o = {po_bin:.4f}, p_e = {pe_bin:.4f}, Cohen の κ = {k_bin:.4f}")
print(f"cohen_kappa_score = {cohen_kappa_score(a_bin, b_bin, labels=['pos', 'neg']):.4f}")

作業者B,pos,neg
作業者A,,
pos,20,5
neg,10,15


p_o = 0.7000, p_e = 0.5000, Cohen の κ = 0.4000
cohen_kappa_score = 0.4000


### 2-2. 第1ラウンド（3値分類：`pos` / `neu` / `neg`）の混同行列と一致率（5-6「品質管理」）

次に、サンプルデータの第1ラウンド（50件）について、作業者 A と作業者 B の一致率を計算します。

In [4]:
labels = ["pos", "neu", "neg"]
r1 = df[df["ラウンド"] == 1].copy()

cm_r1, po_r1, pe_r1, k_r1 = compute_cohen_kappa(r1["作業者A"], r1["作業者B"], labels)
print("第1ラウンドの混同行列（行: 作業者A / 列: 作業者B）:")
display(cm_r1)
print(f"p_o = {po_r1:.4f}, p_e = {pe_r1:.4f}, Cohen の κ = {k_r1:.4f}")
print(f"scikit-learn による検証: κ = {cohen_kappa_score(r1['作業者A'], r1['作業者B'], labels=labels):.4f}")

第1ラウンドの混同行列（行: 作業者A / 列: 作業者B）:


作業者B,pos,neu,neg
作業者A,,,
pos,8,12,6
neu,2,4,0
neg,2,4,12


p_o = 0.4800, p_e = 0.3024, Cohen の κ = 0.2546
scikit-learn による検証: κ = 0.2546


$\kappa \approx 0.2546$ は Landis & Koch の目安では「普通の一致（$0.20 < \kappa \le 0.40$）」にとどまります。この状態では、まだタスク仕様書が十分に固まっていないと考えられます。

---

## 3. 不一致事例の分析と仕様改善・未知データでの再測定（5-2「仕様」、5-6「品質管理」）

### 3-1. 不一致事例とコメントの確認

作業者 A と作業者 B のラベルが食い違った事例と `コメント` 欄を抽出し、どのような点に曖昧さがあったかを確認します。

In [5]:
disagreed_r1 = r1[r1["作業者A"] != r1["作業者B"]].copy()
print(f"第1ラウンドの不一致件数: {len(disagreed_r1)} / {len(r1)} 件\n")

print("不一致パターンの内訳:")
display(disagreed_r1.groupby(["作業者A", "作業者B"]).size().rename("件数").reset_index())

# 主な不一致パターンから事例を表示
sample_disagreed = pd.concat([
    disagreed_r1[(disagreed_r1["作業者A"] == "pos") & (disagreed_r1["作業者B"] == "neu")].head(3),
    disagreed_r1[(disagreed_r1["作業者A"] == "pos") & (disagreed_r1["作業者B"] == "neg")].head(3),
    disagreed_r1[(disagreed_r1["作業者A"] == "neg") & (disagreed_r1["作業者B"] == "neu")].head(2),
])
sample_disagreed[["フレーズID", "作業者A", "作業者B", "極性ラベル", "フレーズ", "コメント"]]

第1ラウンドの不一致件数: 26 / 50 件

不一致パターンの内訳:


,作業者A,作業者B,件数
0,neg,neu,4
1,neg,pos,2
2,neu,pos,2
3,pos,neg,6
4,pos,neu,12


,フレーズID,作業者A,作業者B,極性ラベル,フレーズ,コメント
2,3,pos,neu,neu,駅 の 近く に 飲食店 が 多 く あ る の が,A:立地の良さでpos / B:周辺状況の記述のためneu
3,4,pos,neu,neu,天然 温泉 の 大 浴場 も,A:魅力的な設備でpos / B:名詞句のみで判断保留しneu
7,8,pos,neu,neu,ホテル の すぐ 隣 に コンビニ も あ り 、,A:利便性を評価しpos / B:設備有無の記述のためneu
0,1,pos,neg,pos,大通り 沿い で 車 の 音 は 聞こえ ま す が 二重 窓 の おかげ で 気 に なら...,逆接表現：騒音への言及をBがneg、解消された点をAがposと判定
1,2,pos,neg,pos,外観 は 地味 な 印象 で し た が 内装 は 綺麗 に リニューアル 済み で す,逆接表現：前件のマイナスと後件のプラスで極性が反転
11,12,pos,neg,pos,部屋 は やや 手狭 で し た が 機能 的 に まとめ られ て い て 快適 で,逆接表現：狭さ（neg）と快適さ（pos）の混在で判断が割れた
18,19,neg,neu,neu,提携 駐車 場 が ホテル から 徒歩 ５ 分 ほど 離れ て お り,A:不便とみてneg / B:距離の事実記述のためneu
31,32,neg,neu,neg,備え付け の ドライヤー の 風量 が 弱め だっ た 点,A:マイナス評価としてneg / B:客観的感想としてneu


不一致の原因を見ると、主に次の 2 点で仕様の解釈が割れていたことがわかります。

1. **客観的な設備・立地・プランの記述や短い断片**（例：「徒歩 ３ 分 の 立地」「隣 に コンビニ も あ り 、」「天然 温泉 の 大 浴場 も」）を好条件とみて `pos` とするか、事実・文脈不足として `neu` とするか。
2. **逆接・対比表現**（例：「建物 自体 は 古 め だ が 清掃 が 隅々 まで 行き届い て い る」）で、前件と後件のどちらの極性を代表値とするか。

そこで仕様書を次のように改訂したうえで、**第1ラウンドとは別の未知データ（第2ラウンド 50件）** で再度アノテーションを行いました。

- **改訂ルール 1**: 評価語や主観的感想を含まない客観的な立地・設備・仕様の記述や、文脈から極性が判断できない短い断片フレーズは `neu` とする（「丁寧」「快適」「不満」などの明確な評価表現がある場合のみ `pos` / `neg`）。
- **改訂ルール 2**: 「A だが B」のような逆接・対比を含む場合は、**主節（後件 B）の極性** を採用する。

（※書籍 5-6「品質管理」で述べているとおり、第1ラウンドで議論してすり合わせた後のデータで一致率を計算し直しても「真の一致率」にはならないため、必ず**未知のデータ**で再測定します。）

In [6]:
r2 = df[df["ラウンド"] == 2].copy()
cm_r2, po_r2, pe_r2, k_r2 = compute_cohen_kappa(r2["作業者A"], r2["作業者B"], labels)

print("第2ラウンドの混同行列（仕様書改訂後・未知データ 50件）:")
display(cm_r2)
print(f"p_o = {po_r2:.4f}, p_e = {pe_r2:.4f}, Cohen の κ = {k_r2:.4f}")
print(f"第1・第2ラウンドの比較: κ = {k_r1:.4f} → {k_r2:.4f}")

第2ラウンドの混同行列（仕様書改訂後・未知データ 50件）:


作業者B,pos,neu,neg
作業者A,,,
pos,18,2,0
neu,1,14,1
neg,0,2,12


p_o = 0.8800, p_e = 0.3400, Cohen の κ = 0.8182
第1・第2ラウンドの比較: κ = 0.2546 → 0.8182


---

## 4. 3人以上のアノテーション（Fleiss の $\kappa$）とクラウドソーシングの品質管理（5-2「品質管理」「アノテーションの外注」）

各項目に同じ人数が名義ラベルを付ける場合は **Fleiss の $\kappa$** を用います。
また、クラウドソーシングでは「最初は真面目に作業し、途中から適当にラベルを付ける（あるいは同じ選択肢ばかり選ぶ）**スパマー**」が混じることがあります（書籍 5-2「アノテーションの外注」の脚注）。

ここでは第2ラウンドの 50件を対象に 5 人の作業者（真面目な作業者 1〜4 ＋ 途中から「`pos`」ばかり選ぶ作業者 5）の仮想データを生成し、**事前設定したチェック問題**と**他作業者との一致率**からスパマーを検出・除外してみます。

In [7]:
def fleiss_kappa(ratings, categories):
    """N事例 × k作業者の評価行列から Fleiss の κ を計算する"""
    ratings = np.asarray(ratings)
    n_items, n_raters = ratings.shape
    counts = np.column_stack([(ratings == c).sum(axis=1) for c in categories])
    p_cat = counts.sum(axis=0) / (n_items * n_raters)
    p_e = float(np.sum(p_cat ** 2))
    p_i = (np.sum(counts ** 2, axis=1) - n_raters) / (n_raters * (n_raters - 1))
    p_o = float(np.mean(p_i))
    return float((p_o - p_e) / (1.0 - p_e)), p_o, p_e

# 5人分のクラウドソーシング作業結果をシミュレート
rng = np.random.default_rng(2)
gold_r2 = r2["極性ラベル"].to_numpy()

def simulate_honest(gold, p_correct):
    return np.array([
        y if rng.random() < p_correct else rng.choice([l for l in labels if l != y])
        for y in gold
    ])

workers = {
    "作業者1": simulate_honest(gold_r2, 0.88),
    "作業者2": simulate_honest(gold_r2, 0.86),
    "作業者3": simulate_honest(gold_r2, 0.84),
    "作業者4": simulate_honest(gold_r2, 0.86),
    # 最初の10件は真面目に答え、途中から常に最初の選択肢（pos）だけを選ぶスパマー
    "作業者5（スパマー）": np.array([
        y if i < 10 else "pos" for i, y in enumerate(gold_r2)
    ]),
}

# 全体の中に抜き打ちのチェック問題（答えが既知の事例）を5件混ぜておく
check_idx = np.array([11, 14, 25, 34, 44])

ratings_all = np.column_stack(list(workers.values()))
fk_all, po_all, _ = fleiss_kappa(ratings_all, labels)
print(f"5人全員の Fleiss の κ = {fk_all:.3f} (平均一致割合 P_o = {po_all:.3f})")

# 作業者ごとの品質指標を集計
w_names = list(workers.keys())
summary = []
for w in w_names:
    chk_acc = float((workers[w][check_idx] == gold_r2[check_idx]).mean())
    mean_k = float(np.mean([
        cohen_kappa_score(workers[w], workers[o], labels=labels)
        for o in w_names if o != w
    ]))
    summary.append({
        "作業者": w,
        "チェック問題正答率": round(chk_acc, 2),
        "他作業者との平均κ": round(mean_k, 3),
    })
display(pd.DataFrame(summary))

# スパマーを除外して再計算・多数決で統合
clean_names = ["作業者1", "作業者2", "作業者3", "作業者4"]
ratings_clean = np.column_stack([workers[w] for w in clean_names])
fk_clean, po_clean, _ = fleiss_kappa(ratings_clean, labels)

def majority_vote(mat):
    return np.array([pd.Series(row).mode().iloc[0] for row in mat])

acc_maj_all = accuracy_score(gold_r2, majority_vote(ratings_all))
acc_maj_clean = accuracy_score(gold_r2, majority_vote(ratings_clean))
print(f"スパマー除外後（4人）の Fleiss の κ = {fk_clean:.3f} (P_o = {po_clean:.3f})")
print(f"多数決ラベルの正解率: 全員 {acc_maj_all:.3f} → 除外後 {acc_maj_clean:.3f}")

5人全員の Fleiss の κ = 0.384 (平均一致割合 P_o = 0.606)


,作業者,チェック問題正答率,他作業者との平均κ
0,作業者1,1.0,0.489
1,作業者2,1.0,0.490
2,作業者3,0.8,0.444
3,作業者4,1.0,0.458
4,作業者5（スパマー）,0.2,0.112


スパマー除外後（4人）の Fleiss の κ = 0.589 (P_o = 0.727)
多数決ラベルの正解率: 全員 0.940 → 除外後 0.960


---

## 5. 欠損値や順序尺度を扱える一致率指標：Krippendorff の $\alpha$（5-2 補足）

第 2〜4 節で使った Cohen の $\kappa$ は「作業者 2 人・欠損なし」、Fleiss の $\kappa$ は「すべての項目に同じ人数が回答（欠損なし）・名義尺度」を前提にしています。
しかし実務のクラウドソーシングや 5 段階評価では、次の 2 つの課題がよく起きます。

1. **作業者ごとに一部の事例しかアノテーションしない（疎な評価行列・`NaN` 欠損あり）**
2. **5 段階評価（1〜5）のような順序尺度で、「4 と 5 のズレ」と「1 と 5 の逆転」の重みが異なる**

これらを統一的に扱えるのが **Krippendorff の $\alpha$**（$\alpha = 1 - D_o / D_e$）です。ここでは定義式（一致行列 $o_{ck}$ と尺度別距離関数 $\delta^2(c, k)$）を実装し、次の 2 点を確認します。
- **欠損ありデータでのスパマー検知**（名義尺度 `nominal`）
- **5 段階尺度での名義 $\alpha$ vs 順序 $\alpha$ の違い**（隣接ズレ vs 両極端の誤り）

In [8]:
def krippendorff_alpha(reliability_matrix, level="nominal", value_domain=None):
    """
    items x raters の評価行列（欠損は None / np.nan）から Krippendorff の α を計算する
    level: 'nominal' | 'ordinal' | 'interval'
    """
    mat = pd.DataFrame(reliability_matrix)
    unit_vals = []
    for _, row in mat.iterrows():
        vals = [v for v in row.tolist() if pd.notna(v)]
        if len(vals) >= 2:
            unit_vals.append(vals)
    if not unit_vals:
        return float("nan")

    if value_domain is None:
        value_domain = sorted({v for vals in unit_vals for v in vals})
    val_to_i = {v: i for i, v in enumerate(value_domain)}
    k_len = len(value_domain)

    coinc = np.zeros((k_len, k_len), dtype=float)
    for vals in unit_vals:
        m_u = len(vals)
        for i in range(m_u):
            ci = val_to_i[vals[i]]
            for j in range(m_u):
                if i == j:
                    continue
                cj = val_to_i[vals[j]]
                coinc[ci, cj] += 1.0 / (m_u - 1)

    n_c = coinc.sum(axis=1)
    N = float(n_c.sum())
    if N <= 1:
        return float("nan")

    delta = np.zeros((k_len, k_len), dtype=float)
    for i in range(k_len):
        for j in range(k_len):
            if i == j:
                continue
            if level == "nominal":
                delta[i, j] = 1.0
            elif level == "ordinal":
                lo, hi = min(i, j), max(i, j)
                s = np.sum(n_c[lo:hi + 1]) - (n_c[lo] + n_c[hi]) / 2.0
                delta[i, j] = s ** 2
            elif level == "interval":
                delta[i, j] = float(value_domain[i] - value_domain[j]) ** 2
            else:
                raise ValueError(f"Unknown level: {level}")

    d_o = np.sum(coinc * delta) / N
    d_e = np.sum(np.outer(n_c, n_c) * delta) / (N * (N - 1))
    return 1.0 - float(d_o / d_e) if d_e > 0 else 1.0

# --- 検証 1: 第4節のクラウドソーシングデータに欠損（各作業者が半数強のみ担当）を導入 ---
rng_k = np.random.default_rng(7)
sparse_all = pd.DataFrame({
    w: pd.Series(workers[w], dtype=object).where(rng_k.random(len(gold_r2)) < 0.55)
    for w in w_names
})
missing_rate = float(sparse_all.isna().mean().mean())
alpha_all = krippendorff_alpha(sparse_all, level="nominal", value_domain=labels)
alpha_clean = krippendorff_alpha(sparse_all[clean_names], level="nominal", value_domain=labels)
print(f"疎な評価行列（欠損率 {missing_rate:.1%}）での Nominal α:")
print(f"  スパマー込み（5人）: α = {alpha_all:.3f} → スパマー除外（4人）: α = {alpha_clean:.3f}")

# --- 検証 2: 5段階尺度（1〜5）における Nominal α vs Ordinal α ---
# 不一致件数・発生箇所を完全に揃えた上で「隣接 ±1 のブレ」vs「1↔5 等の大逆転」を比較
gold_5 = r2["極性"].to_numpy()
a_5 = gold_5.copy()
b_adjacent = np.array([
    (v + 1 if v < 5 else 4) if i % 4 == 0 else v
    for i, v in enumerate(gold_5)
])
b_extreme = np.array([
    ({1: 5, 2: 5, 3: 1, 4: 1, 5: 1}[int(v)]) if i % 4 == 0 else v
    for i, v in enumerate(gold_5)
])

ord_rows = []
for name, b_vec in [
    ("ケースA: 隣接ズレ（±1）のみ", b_adjacent),
    ("ケースB: 両極端の逆転（1↔5等）", b_extreme),
]:
    mat_5 = pd.DataFrame({"A": a_5, "B": b_vec})
    ord_rows.append({
        "比較ケース": name,
        "単純一致率": round(float(np.mean(a_5 == b_vec)), 3),
        "Nominal α（完全一致のみ）": round(krippendorff_alpha(mat_5, level="nominal", value_domain=[1,2,3,4,5]), 3),
        "Ordinal α（順序の近さを考慮）": round(krippendorff_alpha(mat_5, level="ordinal", value_domain=[1,2,3,4,5]), 3),
    })
pd.DataFrame(ord_rows)

疎な評価行列（欠損率 48.0%）での Nominal α:
  スパマー込み（5人）: α = 0.477 → スパマー除外（4人）: α = 0.591


,比較ケース,単純一致率,Nominal α（完全一致のみ）,Ordinal α（順序の近さを考慮）
0,ケースA: 隣接ズレ（±1）のみ,0.74,0.666,0.928
1,ケースB: 両極端の逆転（1↔5等）,0.74,0.651,0.320


- **欠損への対応**: クラウドソーシングのように 1 項目あたり一部の作業者しか回答しない疎な行列（欠損率約 48%）でもそのまま計算でき、スパマー除外の効果（$\alpha$ の上昇）を定量評価できます。
- **順序尺度への対応**: ケース A（隣接 ±1 のみ）とケース B（1↔5 の逆転）は単純一致率や Nominal $\alpha$ ではほぼ同じ値に見えますが、Ordinal $\alpha$ は距離に応じたペナルティを与えるため、致命的な両極端の誤りの方を明確に低く評価します。

このように、**「仕様書作成 → 複数人アノテーションと $\kappa$ / Krippendorff の $\alpha$ 計算 → 不一致分析と仕様改訂 → 未知データでの再測定」** のサイクルを回すことで、一貫性の高いアノテーション基準を整備できます。
構築したコーパスを使ったモデル性能・誤り候補検出・能動学習は [機械学習によるコーパス品質管理と能動学習](ch5_corpus_quality.ipynb) を参照してください。